# DeepONet Data Exploration

This notebook explores the dataset generated for training a DeepONet. Targets include potential (phi), velocity (u, v), and pressure (p).

## 1. Load the Dataset

In [ ]:
import numpy as np
import pd = pd
import matplotlib.pyplot as plt
import os

dataset_path = "../dataset_deeponet.npz"
data = np.load(dataset_path, allow_pickle=True)

print("Keys in NPZ:", data.files)

## 2. Summarize Metadata & Performance

We can see basic simulation info and performance profiles.

In [ ]:
metadata = data['metadata']
df = pd.DataFrame(list(metadata))
display(df.head())

print("\nAverage FEM Solve Time: {:.4f}s".format(df['fem_solve_sec'].mean()))

## 3. Deep Dive into Data Structures

In [ ]:
idx = 0
coords = data['mesh_coords'][idx].astype(np.float64)
triangles = data['mesh_triangles'][idx].astype(np.int64)
sdf = data['sdf_sensors'][idx].astype(np.float64)
bc = data['bc_sensors'][idx].astype(np.float64)

print(f"--- Mesh structure for simulation {idx} ---")
print(f"Mesh Coordinates shape: {coords.shape} (N_nodes, 2)")
print(f"Mesh Triangles shape: {triangles.shape} (N_tri, 3)")

print(f"\n--- Sensor structure ---")
print(f"SDF Sensors shape: {sdf.shape} (Matches 32x32 grid)")
print(f"BC Sensors shape: {bc.shape} (Values on outer boundary)")

## 4. Visualizing Mesh vs Sensor Grid (Global & Local Views)

The SDF sensors provide geometry details (even inside the airfoil), while the BC sensors capture the inlet conditions far away.

In [ ]:
fig = plt.figure(figsize=(15, 7))

# Local view (Near Airfoil)
ax1 = fig.add_subplot(1, 2, 1)
ax1.triplot(coords[:, 0], coords[:, 1], triangles, color='gray', lw=0.5, alpha=0.3, label='FEM Mesh')

gx = np.linspace(-1.0, 2.0, 32)
gy = np.linspace(-1.0, 1.0, 32)
GX, GY = np.meshgrid(gx, gy)
sc1 = ax1.scatter(GX, GY, c=sdf, cmap='RdBu_r', s=25, edgecolor='black', lw=0.2, label='SDF Sensors')
plt.colorbar(sc1, ax=ax1, label='SDF Value (Negative=Inside)')

ax1.set_aspect('equal')
ax1.set_xlim(-1.2, 2.2)
ax1.set_ylim(-1.2, 1.2)
ax1.set_title("Local View: Mesh & SDF Sensors")
ax1.legend(loc='upper right')

# Global view (Farfield Conditions)
ax2 = fig.add_subplot(1, 2, 2)
ax2.triplot(coords[:, 0], coords[:, 1], triangles, color='gray', lw=0.2, alpha=0.2)

theta = np.linspace(0, 2*np.pi, 100, endpoint=False)
bx, by = 5.0 * np.cos(theta), 5.0 * np.sin(theta)
sc2 = ax2.scatter(bx, by, c=bc, cmap='viridis', s=15, label='BC Sensors (Dirichlet phi)')
plt.colorbar(sc2, ax=ax2, label='Potential Value (phi)')

# Draw the bounding box of the SDF sensors to show their relative scale
rect = plt.Rectangle((-1, -1), 3, 2, fill=False, color='red', lw=2, label='SDF Grid Area')
ax2.add_patch(rect)

ax2.set_aspect('equal')
ax2.set_title("Global View: BC Sensors at Radius 5.0")
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()

## 5. Continuous Field Visualization

Visualizing the physical results (Potential, Velocity, Pressure) as smooth fields using the mesh connectivity.

In [ ]:
idx = 25
c = data['mesh_coords'][idx].astype(np.float64)
t = data['mesh_triangles'][idx].astype(np.int64)
phi = data['phi_targets'][idx].astype(np.float64)
vel = data['velocity_targets'][idx].astype(np.float64)
pres = data['pressure_targets'][idx].astype(np.float64)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

im0 = axes[0].tricontourf(c[:, 0], c[:, 1], t, phi, levels=50, cmap='viridis')
plt.colorbar(im0, ax=axes[0], label='phi')
axes[0].set_title("Potential")

v_mag = np.linalg.norm(vel, axis=1)
im1 = axes[1].tricontourf(c[:, 0], c[:, 1], t, v_mag, levels=50, cmap='magma')
plt.colorbar(im1, ax=axes[1], label='|V|')
axes[1].set_title("Velocity Magnitude")

im2 = axes[2].tricontourf(c[:, 0], c[:, 1], t, pres, levels=50, cmap='RdBu_r')
plt.colorbar(im2, ax=axes[2], label='Pressure')
axes[2].set_title("Pressure (p)")

for ax in axes:
    ax.set_aspect('equal')
    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(-0.5, 0.5)

plt.tight_layout()
plt.show()